# 05 — Supervised data preparation

This notebook:
- loads the SBS96 matrix and merged clinical data,
- defines the binary smoking target,
- creates train, validation, and test splits at patient level,
- prepares the feature matrices,
- saves the split tables for the model notebook.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

def map_smoking_label(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().lower()

    if value in ["", "nan", "not reported", "unknown"]:
        return "Unknown"
    if "never" in value or "lifelong" in value or "non-smoker" in value or "nonsmoker" in value:
        return "Never"
    if "former" in value or "reformed" in value:
        return "Former"
    if "current" in value:
        return "Current"

    return "Unknown"

## 1. Define paths


In [2]:
sbs96_path = PROJECT_ROOT / "data" / "maf" / "output" / "SBS" / "LUAD.SBS96.all"
clinical_path = PROJECT_ROOT / "data" / "clinical_exposure_merged.tsv"
split_output_dir = PROJECT_ROOT / "data" / "brf_split_binary"

split_output_dir.mkdir(parents=True, exist_ok=True)

## 2. Load the input tables


In [3]:
if sbs96_path is None:
    raise FileNotFoundError("The SBS96 matrix file was not found.")

sbs = pd.read_csv(sbs96_path, sep="\t", index_col=0)
clinical = pd.read_csv(clinical_path, sep="\t")

if sbs.shape[1] != 96 and sbs.shape[0] == 96:
    sbs = sbs.T

sbs_cols = list(sbs.columns)

pd.DataFrame({
    "table": ["SBS matrix", "clinical"],
    "rows": [sbs.shape[0], clinical.shape[0]],
    "columns": [sbs.shape[1], clinical.shape[1]],
})

,table,rows,columns
0,SBS matrix,616,96
1,clinical,508,7


In [4]:
sbs.iloc[:5, :8]

MutationType,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T
TCGA-05-4244-01A-01D-1105-08,5,5,4,2,3,0,1,1
TCGA-05-4249-01A-01D-1105-08,8,8,4,6,3,2,1,0
TCGA-05-4250-01A-01D-1105-08,7,1,9,11,0,2,3,2
TCGA-05-4382-01A-01D-1931-08,50,62,23,40,4,8,3,7
TCGA-05-4384-01A-01D-1753-08,5,1,3,1,0,0,2,0


In [5]:
clinical.head()

,cases.submitter_id,exposures.tobacco_smoking_status,exposures.pack_years_smoked,demographic.age_at_index,demographic.gender,demographic.race,demographic.ethnicity
0,TCGA-62-A471,Current Reformed Smoker for < or = 15 yrs,30.0,64.0,male,white,not hispanic or latino
1,TCGA-67-3773,Current Reformed Smoker for > 15 yrs,NaN,84.0,female,white,not hispanic or latino
2,TCGA-NJ-A7XG,Current Reformed Smoker for > 15 yrs,NaN,49.0,male,black or african american,not hispanic or latino
3,TCGA-91-6848,Current Reformed Smoker for < or = 15 yrs,NaN,59.0,male,white,not hispanic or latino
4,TCGA-55-6986,Lifelong Non-Smoker,NaN,74.0,female,white,NaN


## 3. Merge SBS data with clinical data


In [6]:
sbs = sbs.copy()
sbs["Patient_ID"] = sbs.index.astype(str).str[:12]

sbs_patient = sbs.groupby("Patient_ID", as_index=False)[sbs_cols].mean()
sbs_patient["total_sbs"] = sbs_patient[sbs_cols].sum(axis=1)
sbs_patient = sbs_patient[sbs_patient["total_sbs"] > 0].drop(columns="total_sbs")

clinical = clinical.copy()
clinical = clinical.rename(columns={"cases.submitter_id": "Patient_ID"})
clinical["Patient_ID"] = clinical["Patient_ID"].astype(str).str[:12]

pd.DataFrame({
    "table": ["clinical", "sbs_patient"],
    "rows": [len(clinical), len(sbs_patient)],
    "unique_patient_ids": [
        clinical["Patient_ID"].nunique(),
        sbs_patient["Patient_ID"].nunique(),
    ],
})

,table,rows,unique_patient_ids
0,clinical,508,508
1,sbs_patient,557,557


In [7]:
merged = pd.merge(clinical, sbs_patient, on="Patient_ID", how="inner")

pd.DataFrame({
    "table": ["merged"],
    "rows": [len(merged)],
    "unique_patient_ids": [merged["Patient_ID"].nunique()],
})

,table,rows,unique_patient_ids
0,merged,493,493


In [8]:
merged.head()

,Patient_ID,exposures.tobacco_smoking_status,exposures.pack_years_smoked,demographic.age_at_index,demographic.gender,demographic.race,demographic.ethnicity,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,A[C>T]C,A[C>T]G,A[C>T]T,A[T>A]A,A[T>A]C,A[T>A]G,A[T>A]T,A[T>C]A,A[T>C]C,A[T>C]G,A[T>C]T,A[T>G]A,A[T>G]C,A[T>G]G,A[T>G]T,C[C>A]A,C[C>A]C,C[C>A]G,C[C>A]T,C[C>G]A,C[C>G]C,C[C>G]G,C[C>G]T,C[C>T]A,C[C>T]C,C[C>T]G,C[C>T]T,C[T>A]A,C[T>A]C,C[T>A]G,C[T>A]T,C[T>C]A,C[T>C]C,C[T>C]G,C[T>C]T,C[T>G]A,C[T>G]C,C[T>G]G,C[T>G]T,G[C>A]A,G[C>A]C,G[C>A]G,G[C>A]T,G[C>G]A,G[C>G]C,G[C>G]G,G[C>G]T,G[C>T]A,G[C>T]C,G[C>T]G,G[C>T]T,G[T>A]A,G[T>A]C,G[T>A]G,G[T>A]T,G[T>C]A,G[T>C]C,G[T>C]G,G[T>C]T,G[T>G]A,G[T>G]C,G[T>G]G,G[T>G]T,T[C>A]A,T[C>A]C,T[C>A]G,T[C>A]T,T[C>G]A,T[C>G]C,T[C>G]G,T[C>G]T,T[C>T]A,T[C>T]C,T[C>T]G,T[C>T]T,T[T>A]A,T[T>A]C,T[T>A]G,T[T>A]T,T[T>C]A,T[T>C]C,T[T>C]G,T[T>C]T,T[T>G]A,T[T>G]C,T[T>G]G,T[T>G]T
0,TCGA-62-A471,Current Reformed Smoker for < or = 15 yrs,30.0,64.0,male,white,not hispanic or latino,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,3.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,1.0,1.0,0.0,1.0,2.0,0.0,1.0,1.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,1.0,2.0,4.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,4.0,2.0,1.0,2.0,9.0,3.0,1.0,8.0,10.0,4.0,12.0,19.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,TCGA-67-3773,Current Reformed Smoker for > 15 yrs,NaN,84.0,female,white,not hispanic or latino,3.0,5.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0,3.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,3.0,1.0,0.0,0.0,0.0,0.0,7.0,5.0,3.0,3.0,0.0,0.0,0.0,1.0,2.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,3.0,4.0,3.0,0.0,0.0,0.0,0.0,3.0,2.0,3.0,2.0,0.0,2.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,1.0,1.0,0.0,0.0,0.0,0.0,3.0,4.0,5.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0
2,TCGA-NJ-A7XG,Current Reformed Smoker for > 15 yrs,NaN,49.0,male,black or african american,not hispanic or latino,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,4.0,0.0,5.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,3.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,TCGA-91-6848,Current Reformed Smoker for < or = 15 yrs,NaN,59.0,male,white,not hispanic or latino,11.0,12.0,9.0,5.0,4.0,6.0,5.0,6.0,9.0,4.0,4.0,8.0,1.0,3.0,2.0,0.0,10.0,2.0,7.0,3.0,1.0,2.0,2.0,1.0,26.0,23.0,12.0,29.0,7.0,8.0,4.0,7.0,9.0,10.0,8.0,14.0,0.0,3.0,12.0,3.0,4.0,4.0,6.0,3.0,0.0,2.0,7.0,1.0,14.0,17.0,5.0,5.0,8.0,4.0,5.0,4.0,6.0,6.0,2.0,5.0,2.0,3.0,3.0,2.0,7.0,2.0,2.0,7.0,0.0,0.0,1.0,0.0,12.0,24.0,5.0,16.0,19.0,2.0,3.0,21.0,22.0,10.0,6.0,14.0,1.0,2.0,5.0,0.0,3.0,3.0,2.0,0.0,0.0,0.0,3.0,1.0
4,TCGA-55-6986,Lifelong Non-Smoker,NaN,74.0,female,white,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Define the binary target


In [9]:
merged["Smoking_3"] = merged["exposures.tobacco_smoking_status"].map(map_smoking_label)
merged = merged[merged["Smoking_3"].isin(["Never", "Former", "Current"])].copy()
merged["Smoking_Bin"] = (merged["Smoking_3"] != "Never").astype(int)

display(
    merged["Smoking_3"]
    .value_counts()
    .rename_axis("Smoking_3")
    .reset_index(name="n_rows")
)

(merged["Smoking_Bin"]
.value_counts()
.rename(index={0: "Never", 1: "Ever"})
.rename_axis("Smoking_Bin")
.reset_index(name="n_rows"))

,Smoking_3,n_rows
0,Former,305
1,Current,119
2,Never,69


,Smoking_Bin,n_rows
0,Ever,424
1,Never,69


## 5. Create patient-level train, validation, and test splits


In [10]:
RANDOM_STATE = 42

patient_labels = merged[["Patient_ID", "Smoking_Bin"]].drop_duplicates()

trainval_patients, test_patients = train_test_split(
    patient_labels,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=patient_labels["Smoking_Bin"],
)

train_patients, val_patients = train_test_split(
    trainval_patients,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=trainval_patients["Smoking_Bin"],
)

train_patients.head()

,Patient_ID,Smoking_Bin
426,TCGA-91-6831,1
124,TCGA-69-7974,1
465,TCGA-55-6712,1
77,TCGA-L9-A5IP,1
125,TCGA-55-A4DG,1


In [11]:
train_ids = set(train_patients["Patient_ID"])
val_ids = set(val_patients["Patient_ID"])
test_ids = set(test_patients["Patient_ID"])

pd.DataFrame({
    "check": ["train ∩ val", "train ∩ test", "val ∩ test"],
    "n_overlap": [
        len(train_ids & val_ids),
        len(train_ids & test_ids),
        len(val_ids & test_ids),
    ],
})

,check,n_overlap
0,train ∩ val,0
1,train ∩ test,0
2,val ∩ test,0


## 6. Build row-level split tables and save them


In [12]:
split_output_dir

PosixPath('/Users/michaljendrusak/PycharmProjects/tcga-luad-smoking-mutational-signatures/data/brf_split_binary')

In [13]:
train_df = merged[merged["Patient_ID"].isin(train_ids)].copy()
val_df = merged[merged["Patient_ID"].isin(val_ids)].copy()
test_df = merged[merged["Patient_ID"].isin(test_ids)].copy()

train_df.to_csv(split_output_dir / "train.csv", index=False)
val_df.to_csv(split_output_dir / "val.csv", index=False)
test_df.to_csv(split_output_dir / "test.csv", index=False)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n_rows": [len(train_df), len(val_df), len(test_df)],
    "n_patients": [
        train_df["Patient_ID"].nunique(),
        val_df["Patient_ID"].nunique(),
        test_df["Patient_ID"].nunique(),
    ],
    "n_never": [
        (train_df["Smoking_Bin"] == 0).sum(),
        (val_df["Smoking_Bin"] == 0).sum(),
        (test_df["Smoking_Bin"] == 0).sum(),
    ],
    "n_ever": [
        (train_df["Smoking_Bin"] == 1).sum(),
        (val_df["Smoking_Bin"] == 1).sum(),
        (test_df["Smoking_Bin"] == 1).sum(),
    ],
    "ever_ratio": [
        (train_df["Smoking_Bin"] == 1).sum() / len(train_df),
        (val_df["Smoking_Bin"] == 1).sum() / len(val_df),
        (test_df["Smoking_Bin"] == 1).sum() / len(test_df),
    ],
    "never_ratio": [
        (train_df["Smoking_Bin"] == 0).sum() / len(train_df),
        (val_df["Smoking_Bin"] == 0).sum() / len(val_df),
        (test_df["Smoking_Bin"] == 0).sum() / len(test_df),
    ]
})

split_summary.to_csv(split_output_dir / "split_summary.tsv", sep="\t", index=False)

split_summary

,split,n_rows,n_patients,n_never,n_ever,ever_ratio,never_ratio
0,train,315,315,44,271,0.860317,0.139683
1,validation,79,79,11,68,0.860759,0.139241
2,test,99,99,14,85,0.858586,0.141414
